# RANDOM

In [ ]:

import tsim
import stim
import pyzx as zx
import pyzx_param as param

In [ ]:
circ = tsim.Circuit(
"""
CX 0 1
CX 1 0
X 1
"""
)

In [ ]:
for x in circ:
    print(circ)
    g = circ.pop(index=0)
    x = g.targets_copy()
    circ.append_from_stim_program_text(g.name + " " + str(x[0].qubit_value) + " " + str(x[1].qubit_value))


In [ ]:
for t in range(len(circ)):
    print(1)

In [ ]:
len(circ)

In [ ]:
g = circ.pop(index=0)
print(g.tag)
print(g.name)
x = g.targets_copy()
x

In [ ]:
y = [0,0]

for i in range(0, len(x), 2):
                print(x[i].qubit_value)
                print(x[i + 1].qubit_value)

In [ ]:
z0 = 1
z1 = 0

#calculate probability
denom = abs(z0) ** 2 + abs(z1) ** 2
p = abs(z0) ** 2 / denom

p

In [ ]:
import numpy as np

In [ ]:
had_mat = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

In [ ]:
def _bitstring_to_index(bits: list[int]) -> int:
    """Convert a bitstring [b_0, b_1, ..., b_{n-1}] to an integer index."""
    idx = 0
    for b in bits:
        idx = (idx << 1) | b
    return idx


In [ ]:
n_qubits = 2
state = np.zeros(2 ** n_qubits, dtype=complex)

state[0] = 1
print(state)

target = 0
state = state.reshape(2, n_qubits)
print(state)

# Move target axis to front for easy contraction
state = np.moveaxis(state, target, 0)
state = np.einsum("ij,j...->i...", had_mat, state)
state = np.moveaxis(state, 0, target)
print(state)
state = state.reshape(-1)
print(state)


In [ ]:
x_index = _bitstring_to_index([0,0])
complex(state[x_index])


In [ ]:
def gate_by_gate(circuit: tsim.Circuit):
    circ_final = "I"
    y = [0] * circuit.num_qubits
    for t in range(len(circuit)):
        gate = circ.pop(index=0)
        # Gate is a CNOT so update the output classically
        if gate.name == ("CX" or "CNOT" or "ZCX"):
            targets = gate.targets_copy()
            for i in range(0, len(targets), 2):
                a = targets[i].qubit_value
                b = targets[i + 1].qubit_value
                if y[a] == 1:
                    y[b] = 1 - y[b]
        if gate.name == "H":
    return y

In [ ]:
gate_by_gate(circ)


In [ ]:

import tsim
import stim
import pyzx as zx
import gate_by_gate as gbg
import tsim
import numpy as np
from fractions import Fraction
import pyzx_param as param

In [ ]:

x = [0, 0]

test_circ = tsim.Circuit("""

    H 0
    CX 0 1
    """)
num_qubits = test_circ.num_qubits
# test_circ.append_from_stim_program_text(f"R {' '.join(str(q) for q in range(num_qubits))}")
g = test_circ.diagram("pyzx")

# For each qubit, find the vertex with the highest row number
last_vertices = {}
for v in g.vertices():
    q = g.qubit(v)
    if q not in last_vertices or g.row(v) > g.row(last_vertices[q]):
        last_vertices[q] = v

print(last_vertices)
zx.draw(g, labels=True)

In [ ]:
for qubit, bit in enumerate(x):
    out_vertex = last_vertices[qubit]
    phase = Fraction(0) if bit == 0 else Fraction(1)  # 0 = |0>, pi = |1>
    # Insert a Z-spider with the right phase before the output
    g.set_type(out_vertex, zx.VertexType.Z)
    g.set_phase(out_vertex, phase)

In [ ]:

zx.draw(g, labels=True)

In [ ]:
param.full_reduce(g, paramSafe=True)

In [ ]:
complex(g.scalar.to_number())

In [ ]:
num_qubits = 4
circ_until_now = tsim.Circuit()

circ_until_now.append_from_stim_program_text(f"R {' '.join(str(q) for q in range(num_qubits))}")
circ_until_now.append_from_stim_program_text(f"R {' '.join(str(q) for q in range(num_qubits))}")

circ_until_now

In [ ]:
g = circ_until_now.diagram("pyzx")


In [ ]:
zx.full_reduce(g)
zx.draw(g)

In [ ]:
g.outputs()

# Good stuff

In [1]:
import tsim
import stim
import pyzx as zx
import gate_by_gate as gbg
import tsim
import numpy as np
from fractions import Fraction
import pyzx_param as param
import random

In [ ]:
bell_circ = tsim.Circuit(
    """
    RX 0
    MX 0
    DETECTOR rec[-1]
    """
)

detector_circ = tsim.Circuit(
    """
    R 0
    T 0
    M 0
    T_DAG 0
    """
)

meas = tsim.Circuit(
    """
    R 0 0
    T 0
    M 0
    T_DAG 0
    """
)


big_circ = tsim.Circuit("""
 R 1 14
 RX 4 2 10 12
 R 0 9 16 11 6 3 13 8 7 5 17 15
 CX 2 5 4 7 10 15 12 17
 CX 2 3 5 6 7 8 10 11 12 13 15 16
 CX 2 0 5 9 7 11 10 6 12 8
 CX 4 3 7 6 10 9 12 11 17 16
 T 0
 CX 2 5 4 7 10 15 12 17
 MX 7 5 17 15""")

circ = meas.copy()

circ_zx = circ.diagram("pyzx")

In [ ]:
big = tsim.Circuit("""
 R 1 14
 RX 4 2 10 12
 R 0 9 16 11 6 3 13 8 7 5 17 15
 RX 4 2 10 12 15 14
 R 7 5 17
 CX 2 5 4 7 10 6 12 17 15 16
 CX 4 3 7 6 10 9 12 11 14 15 17 16
 CX 2 3 5 6 7 8 10 11 12 13 16 15
 CX 2 0 5 9 7 11 10 15 12 8
 CX 2 5 4 7 12 17 15 14
 T_DAG 13
 M 17 5 7
 MX 15 10 16 13 2 4 12
 RX 2 4 10
 R 5 7 15
 CX 2 5 4 7 10 15
 CX 2 0 5 9 7 11 10 6
 CX 2 3 5 6 7 8 10 11
 CX 4 3 7 6 10 9 15 14
 CX 0 2 6 10 9 5 11 7
 CX 3 2 6 5 8 7 11 10
 CX 3 4 6 7 9 10 14 15
 CX 2 5 4 7 10 15
 MX 2 4 10
 M 5 7 15
 DETECTOR(3, 0.875, 2, -1, -9) rec[-13] rec[-12] rec[-4]
 RX 15 10 5 2 7 1
 """)
 #
 #
 #
 #


# R 1 14
#  RX 4 2 10 12
#  R 0 9 16 11 6 3 13 8 7 5 17 15
#  CX 2 5 4 7 10 15 12 17
#  CX 2 3 5 6 7 8 10 11 12 13 15 16
#  CX 2 0 5 9 7 11 10 6 12 8
#  CX 4 3 7 6 10 9 12 11 17 16
#  CX 2 5 4 7 10 15 12 17
#  M 7 5 17 15
#  MX 4 2 10 12
#  DETECTOR(2, 2, 0) rec[-8]
#  DETECTOR(2, 0, 0) rec[-7]
#  DETECTOR(4, 3, 0) rec[-6]
#  DETECTOR(4, 1, 0) rec[-5]
#   RX 4 2 10 12 15 14
#  R 7 5 17
#  CX 2 5 4 7 10 6 12 17 15 16
#  CX 4 3 7 6 10 9 12 11 14 15 17 16
#  CX 2 3 5 6 7 8 10 11 12 13 16 15
#  CX 2 0 5 9 7 11 10 15 12 8
#  CX 2 5 4 7 12 17 15 14
#  T_DAG 13
#  M 17 5 7
#  MX 15 10 16 13 2 4 12
#  DETECTOR(4, 3, 1) rec[-10]
#  DETECTOR(2, 0, 1) rec[-9]
#  DETECTOR(2, 2, 1) rec[-8]
#  DETECTOR(4, 1, 1) rec[-7]
#  DETECTOR(4, 2, 1) rec[-5]
#  DETECTOR(3, 1, 1) rec[-5] rec[-6] rec[-12]
#  DETECTOR(1, 0, 1) rec[-3] rec[-13]
#  DETECTOR(1, 2, 1) rec[-2] rec[-14]
#  DETECTOR(3, 3, 1) rec[-1] rec[-11]
#  RX 2 4 10
#  R 5 7 15
#  CX 2 5 4 7 10 15
#  CX 2 0 5 9 7 11 10 6
#  CX 2 3 5 6 7 8 10 11
#  CX 4 3 7 6 10 9 15 14
#  CX 0 2 6 10 9 5 11 7
#  CX 3 2 6 5 8 7 11 10
#  CX 3 4 6 7 9 10 14 15
#  CX 2 5 4 7 10 15
#  MX 2 4 10
#  M 5 7 15
#  DETECTOR(1.25, 0.25, 2, -1, -9) rec[-9] rec[-6]
#  DETECTOR(1.5, 1.875, 2, -1, -9) rec[-8] rec[-5]
#  DETECTOR(1.75, 0.25, 2, -1, -9) rec[-3]
#  DETECTOR(2, 1.875, 2, -1, -9) rec[-2]
#  DETECTOR(3, 0.875, 2, -1, -9) rec[-13] rec[-12] rec[-4]
#  DETECTOR(3.5, 0.875, 2, -1, -9) rec[-1]
#  RX 15 10 5 2 7 1
 # T_DAG 0 3 6 8 9 11 14
 # CX 1 0 2 3 5 6 7 8 10 9 15 14
 # CX 3 1 6 7 10 15
 # CX 6 3 10 11
 # CX 6 10
 # MX 6
 # RX 6
 # CX 6 10
 # CX 6 3 10 11
 # CX 3 1 6 7 10 15
 # CX 1 0 2 3 5 6 7 8 10 9 15 14
 # T 0 3 6 8 9 11 14
 # MX 15 10 5 2 7 1
 # DETECTOR(1.60714, 0.75, 3, -1, -9) rec[-29] rec[-28] rec[-26] rec[-24] rec[-23] rec[-21] rec[-20] rec[-18] rec[-17] rec[-12] rec[-11] rec[-7]
 # DETECTOR(4, 1, 4) rec[-6]
 # DETECTOR(3, 1, 4) rec[-5]
 # DETECTOR(2, 1, 4) rec[-4] rec[-7]
 # DETECTOR(1, 0, 4) rec[-3]
 # DETECTOR(2, 2, 4) rec[-2]
 # DETECTOR(0, 1, 4) rec[-1]

In [ ]:
big2 = tsim.Circuit("""
RX 1 2 3
R 4 5 6
CX 1 4 2 5 3 6
CX 1 5 2 6
RX 7 8
CX 2 4 3 5 7 6 8 1
RX 9 10 11
CX 9 4 10 5 11 7 1 8
CX 11 9 5 10 6 7
RX 12
CX 4 9 7 11
CX 9 11
T_DAG 11
CX 9 11
CX 7 11
RX 13 14
CX 13 9 14 7
CX 9 13 7 14
RX 7
R 6 15
RX 5
R 1
RX 9
R 4 0
CX 14 15 7 6 5 1 9 4 8 0
CX 3 6 14 7 10 5 8 1 13 9 2 4
CX 15 14 2 5 11 9 0 8
CX 11 7 2 6 3 5 0 4
RX 8
CX 7 11 6 2 5 3 4 0
CX 6 3 5 2 9 13 8 0
CX 6 15 5 10 1 8 9 11 4 2
CX 7 6 5 1 9 4 0 8
MX 7

DETECTOR[POST-SELECTION] rec[-1]
M 6
DETECTOR[POST-SELECTION] rec[-1]
M 14
DETECTOR[POST-SELECTION] rec[-1]
MX 5
DETECTOR[POST-SELECTION] rec[-1]
M 1
DETECTOR[POST-SELECTION] rec[-1]
MX 9
DETECTOR[POST-SELECTION] rec[-1]
M 4
DETECTOR[POST-SELECTION] rec[-1]
MX 0
DETECTOR[POST-SELECTION] rec[-1]
RX 9 4 1 5 7 6
T_DAG 2 15 8 10 11 3 13
CX 9 13 4 2 1 8 5 10 7 11 6 15
CX 11 9 5 1 2 6
CX 2 11 5 3
CX 2 5
MX 2
DETECTOR[POST-SELECTION] rec[-1]
RX 2
CX 2 5
CX 2 11 5 3
CX 11 9 5 1 2 6
CX 9 13 4 2 1 8 5 10 7 11 6 15
T 2 15 8 10 11 3 13
MX 9
DETECTOR[POST-SELECTION] rec[-1]
MX 4
DETECTOR[POST-SELECTION] rec[-1]
MX 1
DETECTOR[POST-SELECTION] rec[-1]
MX 5
DETECTOR[POST-SELECTION] rec[-1]
MX 7
DETECTOR[POST-SELECTION] rec[-1]
MX 6
DETECTOR[POST-SELECTION] rec[-1]
""")



In [2]:
tricky_decty = tsim.Circuit(
"""
RX 0
R 1
RX 2
CX 0 1
T 0
H 1
CX 2 1 2 0
MX 2
RX 2
CX 2 1 2 0
MX 2
H 2
DETECTOR rec[-1] rec[-2]
"""
)

In [2]:
t_fuckery = tsim.Circuit(
"""
R 0
RX 1 2
CX 1 0 2 0
MX 1 2
H 1 2
DETECTOR rec[-1] rec[-2]
"""
)

In [ ]:
reseting_again = tsim.Circuit(
"""
RX 0
H 0
H 0
H 0
"""
)

In [3]:
circ = t_fuckery.copy()

circ_zx = circ.diagram("pyzx")


In [4]:
param.draw(circ_zx, labels=True)
print("hello")

hello


In [5]:
circ_splits = gbg.preprocessing(circ)


In [ ]:
big.num_qubits

In [6]:
len(circ_splits)

6

# Buffoonery

In [7]:
param.draw(circ_splits[0])
param.draw(circ_splits[1])
param.draw(circ_splits[2])
param.draw(circ_splits[3])
param.draw(circ_splits[4])
param.draw(circ_splits[5])


In [10]:
g = circ_splits[2].copy()
param.draw(g, labels=True)

In [11]:
g.scalar.print_attrs()


phasenode: [], []
phasevars_pi_pair: []
phasepairs: []
phasevars_halfpi: {}
phase: 0
power2: -2
floatfactor: (1+0j)
approximate_floatfactor: (1+0j)
is_zero: False


'phasenode: [], []\nphasevars_pi_pair: []\nphasepairs: []\nphasevars_halfpi: {}\nphase: 0\npower2: -2\nfloatfactor: (1+0j)\napproximate_floatfactor: (1+0j)\nis_zero: False'

In [ ]:
param.simplify.full_reduce(g, paramSafe=True)
g.scalar.print_attrs()

In [12]:
g.set_phase(6, 0)
param.draw(g, labels=True)

In [13]:
g0 = g.copy()
param.draw(g0, labels=True)

In [14]:
param.simplify.full_reduce(g0, paramSafe=True)
param.draw(g0, labels=True)

In [23]:
g0.scalar.print_attrs()

phasenode: [], []
phasevars_pi_pair: []
phasepairs: [(0, 0, {'c'}, {'a', 'b'})]
phasevars_halfpi: {}
phase: 0
power2: -3
floatfactor: (1+0j)
approximate_floatfactor: (1+0j)
is_zero: False


"phasenode: [], []\nphasevars_pi_pair: []\nphasepairs: [(0, 0, {'c'}, {'a', 'b'})]\nphasevars_halfpi: {}\nphase: 0\npower2: -3\nfloatfactor: (1+0j)\napproximate_floatfactor: (1+0j)\nis_zero: False"

In [26]:
string = {'a': 0, 'b': 0, 'c': 1}

In [27]:
a = g0.scalar.evaluate_scalar(string)
a

(0.7071067811865474+0j)

In [28]:
b = (a / (np.sqrt(2) ** 3))
b

(0.2499999999999999+0j)

In [ ]:
g1 = g.copy()
param.draw(g1, labels=True)
param.simplify.full_reduce(g1, paramSafe=True)
param.draw(g1, labels=True)

In [ ]:
print(g0.scalar)
print(g1.scalar)

In [ ]:
print(g0.scalar.evaluate_scalar({})/ (np.sqrt(2) ** 3))
print(g1.scalar.evaluate_scalar({})/ (np.sqrt(2) ** 3))

In [ ]:
param.simplify.full_reduce(g, paramSafe=True)
print(g.scalar)
param.draw(g)
g.normalize()

In [ ]:
for i in range(len(g.scalar.phasenodes)):
    print(g.scalar.phasenodes[i],"\t",g.scalar.phasenodevars[i])



In [23]:
g.scalar.print_attrs()

phasenode: [], []
phasevars_pi_pair: []
phasepairs: [(0, 0, {'c'}, {'rec[0]', 'b', 'a'})]
phasevars_halfpi: {}
phase: 0
power2: -3
floatfactor: (1+0j)
approximate_floatfactor: (1+0j)
is_zero: False


"phasenode: [], []\nphasevars_pi_pair: []\nphasepairs: [(0, 0, {'c'}, {'rec[0]', 'b', 'a'})]\nphasevars_halfpi: {}\nphase: 0\npower2: -3\nfloatfactor: (1+0j)\napproximate_floatfactor: (1+0j)\nis_zero: False"

# Run it

In [10]:
rng = np.random.default_rng(42)
random.seed(10)
counts = {}
N = 100
detects = None
num = 0
for _ in range(N):
    passed, result, detects = gbg.gate_by_gate(circ, circ_splits, detects)
    if passed:
        num += 1
        key = "".join(map(str, result))
        counts[key] = counts.get(key, 0) + 1

print(f"Samples from {N} runs: {num} passed")
for k, v in sorted(counts.items()):
    if v > 0:
        print(f"  |{k}>: {v} ({100*v/N:.1f}%)")


1, 1, passed
1, 0, y = [1, 1, 1] no


KeyboardInterrupt: 

In [ ]:
detects

# TEST STUF

In [ ]:
bell_circuit = tsim.Circuit(
"""
R 0 1
MX 0 1
DETECTOR rec[-1] rec[-2]
"""
)

In [ ]:
for gate in bell_circuit:
    targets = gate.targets_copy()
    targ_s = ""
    if gate.name == "DETECTOR":
        arr = [" rec"] * len(targets)
        for i, val in enumerate(targets):
            targ_s += arr[i] + f"[{val.value}]"
        print(targ_s)

In [3]:
print("=== Bell-state circuit ===")
# Produces |Φ+> = (|00> + |11>) / sqrt(2)
# Expected: samples should be 00 or 11 with equal probability

bell_circuit = tsim.Circuit(
"""
R 0 1
H 0
CX 0 1
"""
)

# (|000> + |111>) / sqrt(2) — should only see 000 or 111
ghz_circuit = tsim.Circuit(
    """
    H 0
    CX 0 1 0 2
    """
)

x_circuit = tsim.Circuit(
    """
    X 0
    CX 0 1
    """
)

example_circuit = tsim.Circuit(
    """
    H 0
    CX 0 1
    H 1
    CX 0 1
    H 0
    """
)

reset_circuit = tsim.Circuit(
    """
    X 0 1 2 3
    CX 0 1
    R 0
    """
)

resetX_circuit = tsim.Circuit(
    """
    R 0
    H 0
    H 0
    """
)

check_circuit = tsim.Circuit(
    """
    RX 0
    H 0
    H 0
    """
)

measure_circ = tsim.Circuit("""
    RX 0
    MX 0 1
""")

detector_circ = tsim.Circuit(
    """
    RX 0
    MX 0
    DETECTOR rec[-1]
    """
)

circ = resetX_circuit
circ_splits = gbg.preprocessing(circ)

rng = np.random.default_rng(42)
counts = {"00": 0, "11": 0, "01": 0, "10": 0}
N = 100
detects = None
for _ in range(N):
    passed, result, detects = gbg.gate_by_gate(circ.copy(),circ_splits, detects)
    if passed:
        key = "".join(map(str, result))
        counts[key] = counts.get(key, 0) + 1

print(f"Samples from {N} runs:")
for k, v in sorted(counts.items()):
    if v > 0:
        print(f"  |{k}>: {v} ({100*v/N:.1f}%)")


=== Bell-state circuit ===
Samples from 100 runs:
  |0>: 100 (100.0%)


In [ ]:
preproc_circ = tsim.Circuit(
    """
    """
)

In [ ]:
gbg.preprocessing(check_circuit)

In [ ]:
bell_circuit = tsim.Circuit(
"""
R 0
H 0
"""
)

In [ ]:
uggy = param.Graph()

In [ ]:
uggy.add_vertex(ty = zx.VertexType.X)

In [ ]:
uggy.add_vertex(ty = zx.VertexType.Z)
uggy.add_edge([0,1], zx.EdgeType.HADAMARD)

In [ ]:
uggy.add_vertex(ty = zx.VertexType.X)
uggy.add_edge([1,2], zx.EdgeType.SIMPLE)

In [ ]:
uggy.add_params(1, {'a'})

In [ ]:
uggy.add_vertex(ty=zx.VertexType.X)
uggy.add_edge([1, 3], zx.EdgeType.SIMPLE)


In [ ]:
uggy.add_params(0, {'b'})

In [ ]:
uggy.add_vertex(ty=zx.VertexType.Z)
uggy.add_edge([4, 3], zx.EdgeType.HADAMARD)


In [ ]:
param.draw(uggy, labels=True)

In [ ]:
param.full_reduce(uggy)
param.draw(uggy, labels=True)

In [ ]:
print(uggy.scalar.phasevars_halfpi)
print(uggy.scalar.phasevars_pi)
print(uggy.scalar.phasevars_pi_pair)
print(uggy.scalar.phasenodevars)


In [ ]:
dictionary: dict[str, Fraction] = {
    "a": Fraction(0),
    "b": Fraction(1)
}

In [ ]:
uggy.scalar.evaluate_scalar(dictionary)

In [ ]:
g = bell_circuit.diagram("pyzx")


In [ ]:
zx.draw(g, labels=True)

In [ ]:
g.scalar

In [ ]:
bell_circuit = tsim.Circuit(
"""
R 0 1 2
"""
)

In [ ]:
bell_circuit.append_from_stim_program_text(f"T 0")

In [ ]:
g = bell_circuit.diagram("pyzx")
zx.draw(g, labels=True)

In [ ]:
gate = bell_circuit.pop()
targets = gate.targets_copy()
gate.name

In [ ]:
gate.name


In [ ]:
gate.tag == 'T'

In [ ]:
g.outputs

In [ ]:
last_vertices = {}
for v in g.vertices():
    if v.ty == zx.VertexType.BOUNDARY:
        last_vertices[q] = v

In [ ]:
print(last_vertices)
x = [0]

In [ ]:
for qubit, bit in enumerate(x):
    out_vertex = last_vertices[qubit]
    # phase = Fraction(0) if bit == 0 else Fraction(1)  # 0 = |0>, pi = |1>
    # Insert a Z-spider with the right phase before the output
    g.set_type(out_vertex, zx.VertexType.X)
    if qubit == 0: g.add_params(out_vertex, 'a')
    else: g.add_params(out_vertex, 'b')

In [ ]:
param.draw(g, labels=True)

In [ ]:
for x in g.vertices():
    print(x, g.get_params(x))


In [ ]:
param.full_reduce(g, paramSafe=True)
param.draw(g, labels=True)

In [ ]:
g.scalar.phasevars_halfpi
g.scalar.phasevars_pi
g.scalar.phasevars_pi_pair
g.scalar.phasenodevars



In [ ]:
g.scalar.phasevars_pi_pair

In [ ]:
g.scalar.phasevars_pi

In [ ]:
g.scalar.phasenodevars

In [ ]:
'a' in g.scalar.phasenodevars[0]

In [ ]:
00, 01, 10 , 11

In [ ]:
zx.draw(g, labels=True)

In [ ]:
param.full_reduce(g, paramSafe=True)

In [ ]:
amplitude = complex(g.scalar.to_number())
a = amplitude

In [ ]:
a

In [ ]:
amplitude = complex(g.scalar.to_number())
a = amplitude

In [ ]:
x = [0]

In [ ]:
for qubit, bit in enumerate(x):
    out_vertex = last_vertices[qubit]
    phase = Fraction(0) if bit == 0 else Fraction(1)  # 0 = |0>, pi = |1>
    # Insert a Z-spider with the right phase before the output
    g.set_type(out_vertex, zx.VertexType.X)
    g.set_phase(out_vertex, phase)

In [ ]:
bell_circuit = tsim.Circuit(
"""
X_ERROR(0.1) 0
"""
)

In [ ]:
g = bell_circuit.pop(index=0)
g.gate_args_copy()

In [ ]:
zx.draw(g, labels=True)

In [ ]:
for gate in bell_circuit:
    targets = gate.targets_copy()
    print(targets)

In [ ]:
bell_circuit = tsim.Circuit(
"""
R 0
H 0
"""
)

In [ ]:
import string
import itertools

def generate_labels(n, start_label: string = None):
    labels = []
    length = 1
    while len(labels) < n:
        for combo in itertools.product(string.ascii_lowercase, repeat=length):
            m = ''.join(combo)
            if start_label is not None:
                m = start_label + m
            labels.append(m)
            if len(labels) == n:
                break
        length += 1
    return labels

In [ ]:
lbls = generate_labels(30, "n_")
lbls

In [ ]:
circ = tsim.Circuit("""
R 0
M 0
""")
g = circ.diagram("pyzx")

In [ ]:
param.draw(g, labels=True)

In [ ]:
g.remove_vertex(2)

In [ ]:
g.phase(1)

In [ ]:
g.scalar.print_attrs()

In [ ]:
param.full_reduce(g, paramSafe=True)
param.draw(g, labels=True)

In [ ]:
print(g.scalar.phasevars_halfpi)
print(g.scalar.phasevars_pi)
print(g.scalar.phasevars_pi_pair)
print(g.scalar.phasenodevars)